In [34]:
using LowLevelFEM, LinearAlgebra, SparseArrays

openGeometry("torus.geo")

Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Reading 'torus.geo'...
Info    : Meshing 1D...nts                                                                                                                
Info    : [  0%] Meshing curve 6 (Line)
Info    : [ 10%] Meshing curve 7 (Line)
Info    : [ 10%] Meshing curve 8 (Line)
Info    : [ 10%] Meshing curve 9 (Line)
Info    : [ 10%] Meshing curve 10 (Circle)
Info    : [ 10%] Meshing curve 11 (Line)
Info    : [ 20%] Meshing curve 12 (Line)
Info    : [ 20%] Meshing curve 13 (Circle)
Info    : [ 20%] Meshing curve 14 (Line)
Info    : [ 20%] Meshing curve 15 (Circle)
Info    : [ 20%] Meshing curve 16 (Circle)
Info    : [ 20%] Meshing curve 17 (Line)
Info    : [ 30%] Meshing curve 18 (Circle)
Info    : [ 30%] Meshing curve 19 (Extruded)
Info    : [ 30%] Meshing curve 20 (Extruded)
Info    : [ 30%] Meshing curve 21 (Extruded)
Info    : [ 30%] Meshing curve 22 (Extruded)
Info    : [ 30%] Mesh

In [35]:
mat = material("body")
prob = Problem([mat])
Pu = Problem([mat], type=:VectorField, dim=3, field=:u)

Problem("torus", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 26075, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs)

In [44]:
@time K = stiffnessMatrix(prob)

  2.667349 seconds (13.46 M allocations: 5.563 GiB, 17.95% gc time)


sparse([1, 2, 3, 82, 83, 84, 211, 212, 213, 409  …  78174, 78175, 78176, 78177, 78220, 78221, 78222, 78223, 78224, 78225], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  78225, 78225, 78225, 78225, 78225, 78225, 78225, 78225, 78225, 78225], [119613.23002793777, -18555.002072073774, -428.69580004405543, 8047.6857872198325, 7082.447556077612, -298.79654328148854, -23718.308800586114, 7004.818055613767, 1617.8093384153863, -29719.547105826714  …  -25293.20039947651, 22152.02154907486, -3406.071584911351, -90129.23262533995, 6634.888504761908, 3832.945494609376, 68007.77116085812, 26567.932953996944, 7829.456914275557, 293481.9145332813], 78225, 78225)

In [45]:
μ = mat.μ
λ = mat.λ
D = [λ+2μ λ λ 0 0 0; λ λ+2μ λ 0 0 0; λ λ λ+2μ 0 0 0; 0 0 0 μ 0 0; 0 0 0 0 μ 0; 0 0 0 0 0 μ]

@time K = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu))

  2.212343 seconds (1.77 M allocations: 3.782 GiB, 17.20% gc time)


sparse([1, 2, 3, 82, 83, 84, 211, 212, 213, 409  …  78174, 78175, 78176, 78177, 78220, 78221, 78222, 78223, 78224, 78225], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  78225, 78225, 78225, 78225, 78225, 78225, 78225, 78225, 78225, 78225], [119613.23002793774, -18555.002072073774, -428.69580004405725, 8047.6857872198225, 7082.447556077613, -298.79654328148854, -23718.308800586106, 7004.818055613767, 1617.8093384153844, -29719.547105826707  …  -25293.20039947651, 22152.02154907486, -3406.0715849113526, -90129.23262533995, 6634.888504761908, 3832.945494609378, 68007.7711608581, 26567.932953996948, 7829.456914275552, 293481.91453328123], 78225, 78225)

In [38]:
load1 = load("load", fz=5000 / 78.54)
supp = displacementConstraint("supp", ux=0, uy=0, uz=0)

BoundaryCondition("supp", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :uz => 0, :ux => 0))

In [39]:
f = loadVector(prob, [load1])
showDoFResults(f, name="terhelés")

0

In [40]:
q = solveDisplacement(prob, load=[load1], support=[supp], iterative=true)
showDoFResults(q, name="elmozdulás")

1

In [41]:
S = solveStress(q, DoFResults=true)

n1 = normalVector(prob, "mid")
S1 = nodesToElements(S, onPhysicalGroup="mid")

ρn = S1 * n1
σn = n1 * (ρn ⋅ n1)
τn = ρn - σn;

In [42]:
showElementResults(ρn, name="ρₙ")
showElementResults(n1, name="normál vektor")
showElementResults(σn, name="σₙ")
showElementResults(τn, name="τₙ")
sn = showDoFResults(ρn * n1, name="sn")
tn = showDoFResults(τn, name="tn")
plotOnPath("path", sn, name="normál feszültség")
plotOnPath("path", tn, name="csúsztató feszültség")

9

In [43]:
openPostProcessor()

-------------------------------------------------------
Version       : 4.15.2-git
License       : GNU General Public License
Build OS      : Linux64-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack LinuxJoystick MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] Untangle Voro++[contrib] WinslowUntangler Zlib tinyobjloader
FLTK version  : 1.3.8
OCC version   : 7.9.2
Packaged by   : root
Web site      : https://gmsh.info
Issue tracker : https://gitlab.onelab.info/gmsh/gmsh/issues
-------------------------------------------------------
